# Analyzing the results of counterfactual explanations on MNIST


In [ ]:
# import relevant packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns # use deep

In [ ]:
# import results
results = pd.read_csv('results/mnist_full_results_updated.csv', index_col=0)
results.head(10)

In [ ]:
results = results.rename(columns={'correctness': 'success', 'weak_correctness': 'weak_success', 'correctness_ratio': 'success_ratio'})
results

In [ ]:
results[results['method'] == 'alibi-Proto-CF'][results['network'] == 'mnist_cnn_output_100'][results['predicted_label'].notna()]

In [ ]:
# fix correctness and weak correctness for captum
results.loc[results['method'] == 'Captum-MinParamPerturbation', 'success'] = (
    results.loc[results['method'] == 'Captum-MinParamPerturbation']
    .apply(lambda row: 1 if pd.notna(row['predicted_label']) and row['original_predicted_label'] != row['predicted_label'] else 0, axis=1)
)

In [ ]:
results['weak_success'] = results.groupby(['network', 'method', 'image'])['success'].transform('max')

## Question 1: How many instances can the method successfully find an explanation for? 

- Are timeouts equivalent to success?
- Are some methods more successful than others? 
- Does this change per network?
- Are certain decision changes more successful than others?
- Is success correlated with something?

In [ ]:
results[results['timeout'] == 1].groupby(['success', 'method', 'network']).count()['timeout']

In [ ]:
results[results['timeout'] == 0].groupby(['success', 'method', 'network']).count()['timeout']

What does this tell us? For 9 instances of alibi-CF where it did not timeout (i.e. an explanation was found), but that explanation was incorrect. This happens only for mlp_relu_1024. For Captum, there are 73,99,34,2 and 3 examples where it did not time out (i.e. an explanation was found) but the explanation was incorrect. This happens for all except one network (that one is resnet 18). Lastly, the PIECE-paper based methods all seem to have the problem over all networks. Is this a bug in the code?

In [ ]:
results[results['timeout'] == 0][results['method'] == 'alibi-CF'][results['success'] == 0]

In [ ]:
results[results['timeout'] == 0][results['method'] == 'Captum-MinParamPerturbation'][results['success'] == 0].tail(50)

In [ ]:
results[results['timeout'] == 0][results['method'] == 'PIECE'][results['success'] == 0].tail(50)

In [ ]:
# cases where the network was incorrectly classifying images
results[results['original_predicted_label'] != results['original_label']].groupby(['method']).count()

Okay so what I believe may be happening is that either the method legit times out and therefore gives a wring explanation. Or there is some bug somewhere that causes wrong explanations. For PIECE, C-min-Edit and Min-edit this could be the way we check if the prediction is the same as the target. For Captum it usually means it cannot change the original prediction. For alibi-CF, it's all for one particular network and the prediction just can't change. 

In [ ]:
sum_correctness = results.groupby('method')['success'].sum()
count_correctness = results.groupby('method')['success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances (including failed attempts NaNs)
print(percentage_correctness)

In [ ]:
sum_correctness = results.groupby('method')['weak_success'].sum()
count_correctness = results.groupby('method')['weak_success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances (including failed attempts NaNs)
print(percentage_correctness)

In [ ]:
filtered_results = results[(results['timeout'] == 0) | ((results['timeout'] == 1) & (results['success'] == 1))]

In [ ]:
sum_correctness = filtered_results.groupby('method')['success'].sum()
count_correctness = filtered_results.groupby('method')['success'].count()

# Calculate the percentage of correctness
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage
print(percentage_correctness)

In [ ]:
sum_correctness = filtered_results.groupby('method')['weak_success'].sum()
count_correctness = filtered_results.groupby('method')['weak_success'].count()

# Calculate the percentage of correctness
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage
print(percentage_correctness)

So what does this tell us? Overall the alibi methods almost always produce valid explanations if they produce anything at all (no explanation produced). It would be interesting to find out how this changes over the different network architectures next. Since Captum's counterfactual explanation implementation is not targeted and only produces 1 explanation per instance, weak correctness is the same thing as correctness. This also (almost) doesn't change when we remove NaN values (no explanation is produced). This means that Captum is right 64\% of the time. The C-Min-Edit and Min-Edit approaches improve significantly when we account for weak correctness, meaning at least one explanation for that instance is correct. This might be due to some decision boundaries being further apart. Lastly PIECE also improves significantly if we use weak correctness (from 27 to 78\%). 

In [ ]:
# Let's investigate the same question but also group by network
sum_correctness = results.groupby(['method', 'network'])['success'].sum()
count_correctness = results.groupby(['method', 'network'])['success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances (including failed attempts NaNs)
print(percentage_correctness)

df_correctness = percentage_correctness.reset_index()

In [ ]:
sum_correctness = results.groupby(['method', 'network'])['weak_success'].sum()
count_correctness = results.groupby(['method', 'network'])['weak_success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances (including failed attempts NaNs)
print(percentage_correctness)
df_weakcorrectness = percentage_correctness.reset_index()
df_weakcorrectness.rename({'weak_success': 'success'}, axis=1, inplace=True)

In [ ]:
sum_correctness = filtered_results.groupby(['method', 'network'])['success'].sum()
count_correctness = filtered_results.groupby(['method', 'network'])['success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances 
print(percentage_correctness)
df_filtered_correctness = percentage_correctness.reset_index()

In [ ]:
sum_correctness = filtered_results.groupby(['method', 'network'])['weak_success'].sum()
count_correctness = filtered_results.groupby(['method', 'network'])['weak_success'].count()
percentage_correctness = (sum_correctness / count_correctness) * 100

# Display percentage of correct instances 
print(percentage_correctness)
df_filtered_weakcorrectness = percentage_correctness.reset_index()
df_filtered_weakcorrectness.rename({'weak_success': 'success'}, axis=1, inplace=True)

In [ ]:
df_correctness['Source'] = 'success, unfiltered'
df_weakcorrectness['Source'] = 'weak success, unfiltered'
df_filtered_correctness['Source'] = 'success, filtered'
df_filtered_weakcorrectness['Source'] = 'weak success, filtered'

# Combine the DataFrames
df = pd.concat([df_correctness, df_weakcorrectness, df_filtered_correctness, df_filtered_weakcorrectness], ignore_index=True)
df

In [ ]:
df['method'] = df.method.replace(to_replace='Captum-MinParamPerturbation', value='Captum')

In [ ]:
df

In [ ]:
g = sns.catplot(
    data=df, x="method", y="success", hue="Source", col="network",
    kind="bar", height=11, aspect=1, palette="deep"
)
g.set_axis_labels("", "Success")

g.set_titles("{col_name}")
g.despine(left=True)

So what trends do we see per network? For the network named mnist_cnn_output_100, which is the cnn taken from the PIECE paper, we see that Captum cannot find any explanations, whilst if we use weak correctness as a measure, all other methods are 100\% correct. This changes drastically when we are stricter with our definition of correctness. For example, PIECE jumps from less than 50\% correctly explained instances to 100 if you count that at least 1 explanation correct per instance is enough. This implies that these methods only perform well if the decision boundary is 'easy'. Further investigation is needed to confirm this implication. Looking at the network named mnist_output_100, which is a simple mlp and the only network that was not trained by us (the network was sourced from a neural network verification benchmark and the resulting keras and torch weights came from a translated ONNX file). Considering this network is an 'outlier' in this sense, this could be a reason why there is really poor performance using PIECE and alibi-CF (alibi-CF has 0 correct explanations). Another interesting thing to note here is that whilst alibi-CF performs badly here, alibi-Proto-CF performs almost perfectly. This is almost reversed when comparing to mnist_cnn_output_100, where alibi-Proto-CF has very low correctness (it does have high correctness on the explanations it does find). Looking for trends among the other networks, alibi-CF performs better than alibi-Proto-CF on all networks except mnist_cnn_output_100 and mnist_lenet5_output (LeNet5 architecture). 

In [ ]:
results['method'] = results.method.replace(to_replace='Captum-MinParamPerturbation', value='Captum')

In [ ]:
results.loc[(results['method'] == 'Captum') & (results['success'] == 1), 'success_ratio'] = 1

In [ ]:
results[results['method'] == 'Captum'][results['success_ratio'] == 0]

In [ ]:
results[results['method'] == 'Captum'][results['success_ratio']]

In [ ]:
sns.set(rc={'figure.figsize':(10, 6)})
sns.boxplot(x='method', y='success_ratio', data=results, palette='deep')

In [ ]:
sns.set(rc={'figure.figsize':(10, 6)})
sns.violinplot(x='method', y='success_ratio', data=results, palette='deep')

### Sub question on correctness of decision boundaries

In [ ]:
# Add heatmaps next to uncover correctness for each decision boundary per method

grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'target'])['success'].mean()

minedit_correctness = pd.DataFrame(grouped_correctness['Min-Edit'].reset_index()).pivot(index="original_predicted_label", columns="target", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
sns.heatmap(minedit_correctness, cmap="YlGnBu", cbar=True, annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('Target')

# Show the heatmap
plt.show()

In [ ]:
grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'target'])['success'].mean()
cminedit_correctness = pd.DataFrame(grouped_correctness['C-Min-Edit'].reset_index()).pivot(index="original_predicted_label", columns="target", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
sns.heatmap(cminedit_correctness, cmap="YlGnBu", annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('Target')

# Show the heatmap
plt.show()

In [ ]:
grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'target'])['success'].mean()
piece_correctness = pd.DataFrame(grouped_correctness['PIECE'].reset_index()).pivot(index="original_predicted_label", columns="target", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
sns.heatmap(piece_correctness, cmap="YlGnBu", annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('Target')

# Show the heatmap
plt.show()

In [ ]:
grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'target'])['success'].mean()
alibiCF_correctness = pd.DataFrame(grouped_correctness['alibi-CF'].reset_index()).pivot(index="original_predicted_label", columns="target", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
sns.heatmap(alibiCF_correctness, cmap="YlGnBu", cbar=True, annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('Target')

# Show the heatmap
plt.show()

In [ ]:
grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'target'])['success'].mean()
alibiProtoCF_correctness = pd.DataFrame(grouped_correctness['alibi-Proto-CF'].reset_index()).pivot(index="original_predicted_label", columns="target", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
sns.heatmap(alibiProtoCF_correctness, cmap="YlGnBu", cbar=True, annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('Target')

# Show the heatmap
plt.show()

In [ ]:
grouped_correctness = filtered_results.groupby(['method', 'original_predicted_label', 'predicted_label'])['success'].sum()
captum_correctness = pd.DataFrame(grouped_correctness['Captum-MinParamPerturbation'].reset_index()).pivot(index="original_predicted_label", columns="predicted_label", values="success")
plt.figure(figsize=(16, 8))  # Set the figure size
# aptum-MinParamPerturbation
sns.heatmap(captum_correctness, cmap="YlGnBu", cbar=True, annot=True, fmt=".2f")


# Set heatmap title and labels
plt.ylabel('Original Prediction')
plt.xlabel('New Prediction')

# Show the heatmap
plt.show()

## Question 2: are there correlations between correctness and the other metrics?

In [ ]:
results

In [ ]:
sns.scatterplot(data=results, x="success", y="optim_time", hue="method", palette="deep")

In [ ]:
filtered_results['method'] = filtered_results.method.replace(to_replace='Captum-MinParamPerturbation', value='Captum')

## Question 4: How do the different methods perform across different neural architectures with the different metrics?

In [ ]:
# Let's investigate the same question but also group by network
mean_optim_time = results.groupby(['method', 'network'])['optim_time'].mean()
print(mean_optim_time)

df_optimtime = mean_optim_time.reset_index()

In [ ]:
# Let's investigate the same question but also group by network
mean_optim_time = filtered_results.groupby(['method', 'network'])['optim_time'].mean()
print(mean_optim_time)

df_optimtime_filtered = mean_optim_time.reset_index()

In [ ]:
df_optimtime['Source'] = 'unfiltered'
df_optimtime_filtered['Source'] = 'filtered'

# Combine the DataFrames
df = pd.concat([df_optimtime, df_optimtime_filtered], ignore_index=True)
df

In [ ]:
g = sns.catplot(
    data=df, x="method", y="optim_time", 
    kind="box", height=10, aspect=1, palette="deep"
)
g.set_axis_labels("", "Run time")

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
# Let's investigate the same question but also group by network
metrics = ['IM1', 'IM2', 'l1', 'l2', 'linf', 'implausibility', 'morans_i_difference']

for metric in metrics:
    mean_metric = results.groupby(['method', 'network'])[metric].mean().reset_index()
    #mean_metric['method'] = mean_metric.method.replace(to_replace='Captum-MinParamPerturbation', value='Captum')
    g = sns.catplot(
        data=mean_metric, x="method", y=metric, col="network",
        kind="bar", height=10, aspect=1, palette="deep"
    )
    g.set_axis_labels("", metric)

    g.set_titles("{col_name}")
    g.despine(left=True)


## Just investigating Moran's i

In [ ]:
metric = 'morans_i_difference'
only_correct_results = results[results['success'] == 1]
mean_metric = only_correct_results.groupby(['method', 'network'])[metric].mean().reset_index()

g = sns.catplot(
        data=mean_metric, x="method", y=metric, col="network",
        kind="bar", height=10, aspect=1, palette="deep"
    )
g.set_axis_labels("", metric)

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
metric = 'morans_i_difference'
only_correct_results = results[results['success'] == 1]

g = sns.catplot(
        data=only_correct_results, x="method", y=metric,
        kind="violin", height=10, aspect=1, palette="deep"
    )
g.set_axis_labels("", metric)

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
metric = 'morans_i_original_minus_explanation'
only_correct_results = results[results['success'] == 1]

g = sns.catplot(
        data=only_correct_results, x="method", y=metric,
        kind="violin", height=10, aspect=1, palette="deep"
    )
g.set_axis_labels("", metric)

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
metric = 'optim_time'
only_correct_results = results[results['success'] == 1]

g = sns.catplot(
        data=only_correct_results, x="method", y=metric,
        kind="box", height=10, aspect=1, palette="deep"
    )
g.set_axis_labels("", metric)

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
metric = 'morans_i_explanation'
only_correct_results = results[results['success'] == 1]
mean_metric = only_correct_results.groupby(['method', 'network'])[metric].mean().reset_index()

g = sns.catplot(
        data=mean_metric, x="method", y=metric, col="network",
        kind="bar", height=10, aspect=1, palette="deep"
    )
g.set_axis_labels("", metric)

g.set_titles("{col_name}")
g.despine(left=True)

In [ ]:
only_correct_results = filtered_results[filtered_results['success'] == 1]

In [ ]:
# Let's investigate the same question but also group by network
mean_moransi = only_correct_results.groupby(['network', 'method'])['morans_i_difference'].mean()
print(mean_moransi)

df_moransi = mean_moransi.reset_index()